In [8]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv('diabetes.csv')

In [10]:
df.head(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0


In [11]:
df.shape

(768, 9)

In [12]:
from sklearn.ensemble import RandomForestClassifier

In [13]:
rf = RandomForestClassifier()

In [24]:
df.Glucose = np.where(df.Glucose==0, df.Glucose.median(), df.Glucose)
df.Insulin = np.where(df.Insulin==0, df.Insulin.median(), df.Insulin)
df.SkinThickness = np.where(df.SkinThickness==0, df.SkinThickness.median(), df.SkinThickness)

In [25]:
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
296,2,146.0,70,38.0,360.0,28.0,0.337,29,1
712,10,129.0,62,36.0,30.5,41.2,0.441,38,1
482,4,85.0,58,22.0,49.0,27.8,0.306,28,0
628,5,128.0,80,23.0,30.5,34.6,0.144,45,0
474,4,114.0,64,23.0,30.5,28.9,0.126,24,0


In [26]:
X=df.drop(columns=['Outcome'], axis=1)
y=df.Outcome

In [27]:
X.head(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148.0,72,35.0,30.5,33.6,0.627,50
1,1,85.0,66,29.0,30.5,26.6,0.351,31
2,8,183.0,64,23.0,30.5,23.3,0.672,32
3,1,89.0,66,23.0,94.0,28.1,0.167,21
4,0,137.0,40,35.0,168.0,43.1,2.288,33


In [28]:
from sklearn.model_selection import train_test_split as tts

In [29]:
X_train,X_test,y_train,y_test=tts(X,y,test_size=0.2, random_state=0)

In [30]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=10).fit(X_train,y_train)

In [31]:
pred = rf.predict(X_test)

In [32]:
y.value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [33]:
from sklearn.metrics import confusion_matrix, accuracy_score,classification_report

In [34]:
print(confusion_matrix(y_test,pred))

[[95 12]
 [21 26]]


In [35]:
accuracy_score(y_test,pred)

0.7857142857142857

In [36]:
classification_report(y_test,pred)

'              precision    recall  f1-score   support\n\n           0       0.82      0.89      0.85       107\n           1       0.68      0.55      0.61        47\n\n    accuracy                           0.79       154\n   macro avg       0.75      0.72      0.73       154\nweighted avg       0.78      0.79      0.78       154\n'

In [37]:
 model = RandomForestClassifier(n_estimators=300, criterion='entropy',
                               max_features='sqrt',
                               min_samples_leaf=10,
                               random_state=0)

In [39]:
model.fit(X_train,y_train)

RandomForestClassifier(criterion='entropy', min_samples_leaf=10,
                       n_estimators=300, random_state=0)

In [40]:
pred2 = model.predict(X_test)

In [41]:
confusion_matrix(y_test,pred2)

array([[98,  9],
       [17, 30]], dtype=int64)

In [42]:
accuracy_score(y_test,pred2)

0.8311688311688312

In [43]:
classification_report(y_test,pred2)

'              precision    recall  f1-score   support\n\n           0       0.85      0.92      0.88       107\n           1       0.77      0.64      0.70        47\n\n    accuracy                           0.83       154\n   macro avg       0.81      0.78      0.79       154\nweighted avg       0.83      0.83      0.83       154\n'

In [45]:
# Always random search CV first to narrow down option 
# Then we may learn Grid Search CV

In [46]:
from sklearn.model_selection import RandomizedSearchCV

In [49]:
n_estimators=[int(x) for x in np.linspace(start=200,stop=2000,num=10)]

In [50]:
n_estimators

[200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

In [51]:
max_features = ['auto', 'sqrt', 'log2']

In [52]:
max_depth=[int(x) for x in np.linspace(10,1000,10)]

In [54]:
min_samples_split=[2,4,10,14]

In [56]:
min_samples_leaf=[1,2,4,6,8]

In [57]:
random_grid={
    'n_estimators': n_estimators,
    'max_features':max_features,
    'max_depth':max_depth,
    'min_samples_split':min_samples_split,
    'min_samples_leaf':min_samples_leaf,
    'criterion':['entropy', 'gini']
}

In [58]:
rf = RandomForestClassifier()

In [59]:
rsv=RandomizedSearchCV(estimator=rf,
                      param_distributions=random_grid,
                      n_iter=100,
                      cv=3,
                      verbose=2,
                      random_state=100,
                      n_jobs=-100)

In [60]:
rsv.fit(X_train,y_train)

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END criterion=entropy, max_depth=1000, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time=   0.1s
[CV] END criterion=entropy, max_depth=1000, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time=   0.1s
[CV] END criterion=entropy, max_depth=1000, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time=   0.1s
[CV] END criterion=gini, max_depth=120, max_features=sqrt, min_samples_leaf=6, min_samples_split=4, n_estimators=1400; total time=   1.3s
[CV] END criterion=gini, max_depth=120, max_features=sqrt, min_samples_leaf=6, min_samples_split=4, n_estimators=1400; total time=   1.3s
[CV] END criterion=gini, max_depth=120, max_features=sqrt, min_samples_leaf=6, min_samples_split=4, n_estimators=1400; total time=   1.3s
[CV] END criterion=gini, max_depth=340, max_features=sqrt, min_samples_leaf=1, min_samples_split=14,

[CV] END criterion=entropy, max_depth=1000, max_features=sqrt, min_samples_leaf=8, min_samples_split=2, n_estimators=2000; total time=   2.0s
[CV] END criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=14, n_estimators=200; total time=   0.1s
[CV] END criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=14, n_estimators=200; total time=   0.1s
[CV] END criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=14, n_estimators=200; total time=   0.1s
[CV] END criterion=entropy, max_depth=450, max_features=auto, min_samples_leaf=1, min_samples_split=14, n_estimators=1000; total time=   0.0s
[CV] END criterion=entropy, max_depth=450, max_features=auto, min_samples_leaf=1, min_samples_split=14, n_estimators=1000; total time=   0.0s
[CV] END criterion=entropy, max_depth=450, max_features=auto, min_samples_leaf=1, min_samples_split=14, n_estimators=1000; total time=   0.0s
[CV] END criterion=en

[CV] END criterion=entropy, max_depth=560, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=1800; total time=   2.6s
[CV] END criterion=entropy, max_depth=560, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=1800; total time=   2.2s
[CV] END criterion=entropy, max_depth=10, max_features=auto, min_samples_leaf=1, min_samples_split=2, n_estimators=600; total time=   0.0s
[CV] END criterion=entropy, max_depth=10, max_features=auto, min_samples_leaf=1, min_samples_split=2, n_estimators=600; total time=   0.0s
[CV] END criterion=entropy, max_depth=10, max_features=auto, min_samples_leaf=1, min_samples_split=2, n_estimators=600; total time=   0.0s
[CV] END criterion=entropy, max_depth=120, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=2000; total time=   2.1s
[CV] END criterion=entropy, max_depth=120, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=2000; total time=   2.2s
[CV] END criterion=

[CV] END criterion=entropy, max_depth=230, max_features=log2, min_samples_leaf=6, min_samples_split=4, n_estimators=1800; total time=   2.0s
[CV] END criterion=entropy, max_depth=230, max_features=log2, min_samples_leaf=6, min_samples_split=4, n_estimators=1800; total time=   2.1s
[CV] END criterion=entropy, max_depth=230, max_features=log2, min_samples_leaf=6, min_samples_split=4, n_estimators=1800; total time=   2.0s
[CV] END criterion=entropy, max_depth=780, max_features=auto, min_samples_leaf=8, min_samples_split=2, n_estimators=2000; total time=   0.0s
[CV] END criterion=entropy, max_depth=780, max_features=auto, min_samples_leaf=8, min_samples_split=2, n_estimators=2000; total time=   0.0s
[CV] END criterion=entropy, max_depth=780, max_features=auto, min_samples_leaf=8, min_samples_split=2, n_estimators=2000; total time=   0.0s
[CV] END criterion=gini, max_depth=670, max_features=auto, min_samples_leaf=6, min_samples_split=10, n_estimators=400; total time=   0.0s
[CV] END criteri

[CV] END criterion=entropy, max_depth=780, max_features=sqrt, min_samples_leaf=8, min_samples_split=14, n_estimators=400; total time=   0.4s
[CV] END criterion=gini, max_depth=450, max_features=log2, min_samples_leaf=1, min_samples_split=14, n_estimators=1200; total time=   1.4s
[CV] END criterion=gini, max_depth=450, max_features=log2, min_samples_leaf=1, min_samples_split=14, n_estimators=1200; total time=   1.4s
[CV] END criterion=gini, max_depth=450, max_features=log2, min_samples_leaf=1, min_samples_split=14, n_estimators=1200; total time=   1.4s
[CV] END criterion=entropy, max_depth=450, max_features=log2, min_samples_leaf=6, min_samples_split=2, n_estimators=400; total time=   0.4s
[CV] END criterion=entropy, max_depth=450, max_features=log2, min_samples_leaf=6, min_samples_split=2, n_estimators=400; total time=   0.4s
[CV] END criterion=entropy, max_depth=450, max_features=log2, min_samples_leaf=6, min_samples_split=2, n_estimators=400; total time=   0.5s
[CV] END criterion=gin

[CV] END criterion=gini, max_depth=670, max_features=log2, min_samples_leaf=1, min_samples_split=14, n_estimators=1600; total time=   1.8s
[CV] END criterion=gini, max_depth=670, max_features=log2, min_samples_leaf=1, min_samples_split=14, n_estimators=1600; total time=   1.9s
[CV] END criterion=entropy, max_depth=890, max_features=auto, min_samples_leaf=4, min_samples_split=14, n_estimators=1000; total time=   0.0s
[CV] END criterion=entropy, max_depth=890, max_features=auto, min_samples_leaf=4, min_samples_split=14, n_estimators=1000; total time=   0.0s
[CV] END criterion=entropy, max_depth=890, max_features=auto, min_samples_leaf=4, min_samples_split=14, n_estimators=1000; total time=   0.0s


C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
105 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
105 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\dell\anaconda3\Lib\site-packages\sk

RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(), n_iter=100,
                   n_jobs=-100,
                   param_distributions={'criterion': ['entropy', 'gini'],
                                        'max_depth': [10, 120, 230, 340, 450,
                                                      560, 670, 780, 890,
                                                      1000],
                                        'max_features': ['auto', 'sqrt',
                                                         'log2'],
                                        'min_samples_leaf': [1, 2, 4, 6, 8],
                                        'min_samples_split': [2, 4, 10, 14],
                                        'n_estimators': [200, 400, 600, 800,
                                                         1000, 1200, 1400, 1600,
                                                         1800, 2000]},
                   random_state=100, verbose=2)

In [61]:
rsv.best_params_

{'n_estimators': 1400,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'max_depth': 1000,
 'criterion': 'entropy'}

In [62]:
rsv.best_score_

0.7556830862426271

## Automated Hyper-parameter Tuning

## Bayesian Optimazation

In [63]:
!pip install hyperopt

  Obtaining dependency information for hyperopt from https://files.pythonhosted.org/packages/b6/cd/5b3334d39276067f54618ce0d0b48ed69d91352fbf137468c7095170d0e5/hyperopt-0.2.7-py2.py3-none-any.whl.metadata
  Obtaining dependency information for py4j from https://files.pythonhosted.org/packages/10/30/a58b32568f1623aaad7db22aa9eafc4c6c194b429ff35bdc55ca2726da47/py4j-0.10.9.7-py2.py3-none-any.whl.metadata
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
    --------------------------------------- 0.0/1.6 MB 660.6 kB/s eta 0:00:03
   - -------------------------------------- 0.0/1.6 MB 393.8 kB/s eta 0:00:04
   -- ------------------------------------- 0.1/1.6 MB 744.7 kB/s eta 0:00:03
   -- ------------------------------------- 0.1/1.6 MB 590.8 kB/s eta 0:00:03
   ----- ---------------------------------- 0.2/1.6 MB 981.9 kB/s eta 0:00:02
   ------ --------------------------------- 0.3/1.6 MB 923.9 kB/s eta 0:00:02
   ----------- ---------------------------- 0.5/1.6 MB 1.4

In [64]:
from hyperopt import hp,fmin,tpe,STATUS_OK,Trials

In [74]:
space = {'criterion': hp.choice('criterion', ['entropy', 'gini']),
        'max_depth': hp.quniform('max_depth', 1, 50, 1),
        'max_features': hp.choice('max_features', ['auto', 'sqrt','log2', None]),
        'min_samples_leaf': hp.uniform('min_samples_leaf', 0, 0.5),
        'min_samples_split' : hp.uniform ('min_samples_split', 0, 1),
        'n_estimators' : hp.choice('n_estimators', [10, 50, 300, 750, 1200,1300,1500])
    }

In [75]:
space

{'criterion': <hyperopt.pyll.base.Apply at 0x1b13cc67ad0>,
 'max_depth': <hyperopt.pyll.base.Apply at 0x1b13cc65190>,
 'max_features': <hyperopt.pyll.base.Apply at 0x1b13cc66950>,
 'min_samples_leaf': <hyperopt.pyll.base.Apply at 0x1b13ca43f90>,
 'min_samples_split': <hyperopt.pyll.base.Apply at 0x1b13ca6b090>,
 'n_estimators': <hyperopt.pyll.base.Apply at 0x1b13ca44850>}

In [76]:
def objective(space):
    model = RandomForestClassifier(criterion = space['criterion'], max_depth = space['max_depth'],
                                 max_features = space['max_features'],
                                 min_samples_leaf = space['min_samples_leaf'],
                                 min_samples_split = space['min_samples_split'],
                                 n_estimators = space['n_estimators'], 
                                 )
    
    accuracy = cross_val_score(model, X_train, y_train, cv = 5).mean()

    # We aim to maximize accuracy, therefore we return it as a negative value
    return {'loss': -accuracy, 'status': STATUS_OK }

In [77]:
from sklearn.model_selection import cross_val_score
trials = Trials()
best = fmin(fn= objective,
            space= space,
            algo= tpe.suggest,
            max_evals = 80,
            trials= trials)

  0%|                               | 0/80 [00:00<?, ?trial/s, best loss=?]

job exception: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validation.InvalidParameterError: Th

  0%|                               | 0/80 [00:00<?, ?trial/s, best loss=?]


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validation.InvalidParameterError: The '